# 1. Spark (General) Performance Optimization

### 1.1 Data Ingestion & File Size
- Problem: Too many small files or very large skewed files
- Best practices:
   - Prefer Parquet / Delta over CSV/JSON
   - Ideal file size: 128 MB – 1 GB
  - Avoid reading millions of tiny files
- Why: Columnar format + predicate pushdown + compressio
> spark.read.parquet("/data/path")

### 1.2 Partitioning Strategy
### a) Input partitions

Controlled by:
- spark.sql.files.maxPartitionBytes (default 128 MB)
- spark.sql.shuffle.partitions (default 200 ❌)

> spark.conf.set("spark.sql.shuffle.partitions", 200)  # tune based on cluster

### b) Repartition vs Coalesce
Rule:
  - Wide transformation → repartition
  - Just reducing partitions → coalesce

> df = df.repartition(200)   # full shuffle (increase or rebalance)

> df = df.coalesce(50)       # no shuffle (reduce partitions)

### 1.3 Avoid Unnecessary Shuffles
Shuffles are the #1 performance killer

Avoid:
- groupBy on high-cardinality columns
- Unnecessary distinct
- Repartition before join

Prefer:
> df.groupBy("country").count()

Instead of:
> df.select(col('country')).distinct().count()

### 1.4 Join Optimization
### a) Broadcast Join
When: small table < 100–500 MB
>   from pyspark.sql.functions import broadcast

>   df_large.join(broadcast(df_small), "id")

### b) Avoid Cartesian Joins
> spark.conf.set("spark.sql.crossJoin.enabled", "false")

### 1.5 Caching / Persistence

Cache only when reused multiple times

> df.cache()

> df.count()   # action triggers cache

Storage levels:

- MEMORY_ONLY
- MEMORY_AND_DISK (safer)

❌ Don’t cache large DataFrames used once

### 1.6 Actions Trigger Jobs

Actions:

> show()

> count()

> write()

> collect()

Multiple actions = multiple jobs
👉 Cache if reused

### 1.7 Data Skew Handling

Symptoms:

One task runs much longer

Executor OOM

Solutions:

Salting

> from pyspark.sql.functions import rand

> df = df.withColumn("salt", (rand()*10).cast("int"))


AQE (Adaptive Query Execution) – Spark 3+

> spark.conf.set("spark.sql.adaptive.enabled", "true")


# 2. Delta Lake Optimization

### 2.1 Use Delta Instead of Parquet

Why Delta?

- ACID transactions
- Schema enforcement
- Time travel
- Optimized MERGE/UPDATE

### 2.2 Optimize File Size (Small File Problem)
### a) Auto Optimize (Databricks)
  ALTER TABLE sales SET TBLPROPERTIES (
    
    delta.autoOptimize.optimizeWrite = true,
    
    delta.autoOptimize.autoCompact = true
  
  )


✅ Once enabled at table level → no need to specify in write options

### b) OPTIMIZE Command
> **OPTIMIZE sales;**

With partition pruning:

>  **OPTIMIZE sales WHERE date >= '2025-01-01';**

### 2.3 Z-ORDER for Faster Queries
> OPTIMIZE sales ZORDER BY (customer_id, order_id);

Use when:
Frequently filtering on non-partition columns

### 2.4 Partitioning Delta Tables (Very Important)
**Good partition columns:**

Low to medium cardinality

Used in filters

Examples:

**date**

**country**

> df.write.format("delta").partitionBy("date").save("/delta/sales")

### 2.5 Efficient MERGE (Incremental Loads)
>   **MERGE INTO target t
> 
>   USING source s
>   
>   ON t.id = s.id
>   
>   WHEN MATCHED THEN UPDATE SET *
>   
>   WHEN NOT MATCHED THEN INSERT ***

Optimize MERGE:
- Filter source data (only changed records)
- Partition target table
- Enable Z-ORDER on join keys

### 2.6 Vacuum (Storage Cleanup)
> **VACUUM sales RETAIN 168 HOURS;**

⚠️ Never reduce retention without understanding time-travel impact

# 3. Databricks-Specific Optimizations

### 3.1 Cluster Optimization
**Use the right cluster**:

> Job clusters → batch jobs

> All-purpose/Serverless clusters → dev/debug

Enable:

- Autoscaling

- Photon (if available)

### 3.2 Photon Engine
✔ Faster SQL, joins, aggregations
✔ No code changes required

Enable via cluster settings

### 3.3 Delta Cache
- Caches data on local SSD
- Automatic for Delta tables

Best for:
- Repeated reads
- BI workloads

### 3.4 Unity Catalog + Volumes

- Better metadata performance
- Secure data access
- Works seamlessly with Delta optimizations

### 3.5 Workflows & Scheduling

- Avoid overlapping heavy jobs
- Schedule OPTIMIZE during low-traffic windows
- Separate ingestion and transformation jobs

# Streaming + Incremental Optimization

### 4.1 Auto Loader (Databricks)
df = (spark.readStream.format("cloudFiles")\            
            .option("cloudFiles.format", "json")\            
            .load("/raw"))

Optimizations:
- cloudFiles.maxFilesPerTrigger
- Schema evolution enabled

### 4.2 Delta Streaming Writes
df.writeStream.format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/chk") \
  .start("/delta/target")

| Area         | Optimization                     |
| ------------ | -------------------------------- |
| File format  | Use Delta/Parquet                |
| Partitions   | Tune shuffle partitions          |
| Joins        | Broadcast small tables           |
| Shuffles     | Minimize wide transformations    |
| Caching      | Cache reused DF only             |
| Skew         | Salting + AQE                    |
| Delta writes | Auto Optimize                    |
| Reads        | Z-ORDER                          |
| MERGE        | Filter source + partition target |
| Databricks   | Photon + Job clusters            |
